# NINAAD WAGLE | BTECH AI SEM V | I065 B2 | NLP LAB 8

A **word embedding** turns every word into a short list of numbers, called a vector. Words that are used
in similar places get similar vectors, so the computer can tell that *team* and *player* are related
while *team* and *flour* are not.

**Word2Vec** learns these vectors by reading text and guessing which words appear next to each other.
Our documents are three short paragraphs: two about sports and one about baking. We already know
which two should come out as similar, so we can check the result.

# Part A : To implement word embedding using Word2Vec

## Step 1 : Preparing the three documents

Each paragraph goes through `preprocess()` from `preprocessing.py`. It lowercases the text, splits
it into words, and removes punctuation and stopwords (*the, is, of*). Lemmatizing changes *players*
into *player*, so the same word is not counted as two different ones.

In [1]:
import sys
sys.path.append("..")   # preprocessing.py lives one level up, in NLP-Lab-Codes/

import pandas as pd
from gensim.models import Word2Vec
from preprocessing import preprocess, document_vector, cosine_similarity

docs = [
    "Cricket is played between two teams of eleven players. The batting team tries to score runs by hitting "
    "the ball, while the bowling team tries to get the batsmen out. A match can last one day or five days, "
    "and the team with more runs wins the game.",
    "Football is played between two teams of eleven players on a large field. Players kick the ball and try "
    "to score goals against the other team. A match lasts ninety minutes, and the team that scores more goals "
    "wins the game.",
    "To bake bread, mix flour, water, yeast and salt into a soft dough. Knead the dough and leave it in a warm "
    "place so the yeast can make it rise. Then shape the dough and bake it in a hot oven until the bread turns "
    "golden brown.",
]
doc_labels = ["Doc 1 (cricket)", "Doc 2 (football)", "Doc 3 (baking)"]

tokens = [preprocess(doc, lemmatize=True) for doc in docs]
for label, t in zip(doc_labels, tokens):
    print(label, ":", t)

Doc 1 (cricket) : ['cricket', 'played', 'two', 'team', 'eleven', 'player', 'batting', 'team', 'try', 'score', 'run', 'hitting', 'ball', 'bowling', 'team', 'try', 'get', 'batsman', 'match', 'last', 'one', 'day', 'five', 'day', 'team', 'run', 'win', 'game']
Doc 2 (football) : ['football', 'played', 'two', 'team', 'eleven', 'player', 'large', 'field', 'player', 'kick', 'ball', 'try', 'score', 'goal', 'team', 'match', 'last', 'ninety', 'minute', 'team', 'score', 'goal', 'win', 'game']
Doc 3 (baking) : ['bake', 'bread', 'mix', 'flour', 'water', 'yeast', 'salt', 'soft', 'dough', 'knead', 'dough', 'leave', 'warm', 'place', 'yeast', 'make', 'rise', 'shape', 'dough', 'bake', 'hot', 'oven', 'bread', 'turn', 'golden', 'brown']


## Step 2 : Training a Skip-gram Word2Vec model with gensim

We use **Skip-gram**. The model takes one word and tries to guess the words around it, up to `window`
words on each side. After every wrong guess the vectors are adjusted a little. After many rounds,
words that share neighbours end up with similar vectors.

**CBOW** (Continuous Bag of Words) works the other way round: it looks at the surrounding words and
guesses the word in the middle. CBOW trains faster, but Skip-gram works better on small texts and
rare words, which is why we picked it. In gensim the only change is `sg=0` instead of `sg=1`.

- `vector_size=50` : every word becomes a list of 50 numbers
- `window=3` : up to 3 words on each side count as neighbours
- `min_count=1` : keep every word, because our text is very small
- `epochs=300` : read the text 300 times, because a short text has little to learn from
- `seed=42, workers=1` : gives the same result on every run

In [2]:
model = Word2Vec(tokens, vector_size=50, window=3, min_count=1, sg=1, epochs=300, seed=42, workers=1)

print("vocabulary size       :", len(model.wv))
print("vector for 'team'     :", model.wv["team"][:5].round(3), "... (50 numbers)")
print("closest to 'team'     :", model.wv.most_similar("team", topn=3))
print("closest to 'dough'    :", model.wv.most_similar("dough", topn=3))

vocabulary size       : 50
vector for 'team'     : [ 0.096 -0.172 -0.077  0.005  0.144] ... (50 numbers)
closest to 'team'     : [('try', 0.9953668713569641), ('last', 0.9943561553955078), ('score', 0.9935961365699768)]
closest to 'dough'    : [('yeast', 0.9972583055496216), ('place', 0.9970195293426514), ('leave', 0.9969326257705688)]


# Part B : To find similarity between two documents using Word2Vec

## Step 1 : Averaging word vectors and comparing documents with cosine similarity

A document has many words, but we need one vector for the whole document. The simplest way is to
take the average of all its word vectors (`document_vector`).

We then compare two document vectors with **cosine similarity**, which measures the angle between them.
A score of 1 means they point the same way, so the documents are very similar. A score of 0 means
they are unrelated.

Our text is tiny, so every score comes out high. The order is what matters: the two sports paragraphs
score higher with each other than either does with the baking one. gensim's own `n_similarity` does the
same averaging, and it gives the same number as our function.

In [3]:
doc_vectors = [document_vector(t, model.wv) for t in tokens]

similarity = pd.DataFrame([[round(cosine_similarity(a, b), 4) for b in doc_vectors] for a in doc_vectors],
                          index=doc_labels, columns=doc_labels)

print("Doc 1 vs Doc 2 :", round(cosine_similarity(doc_vectors[0], doc_vectors[1]), 4),
      "| gensim n_similarity :", round(float(model.wv.n_similarity(tokens[0], tokens[1])), 4))
similarity

Doc 1 vs Doc 2 : 0.9991 | gensim n_similarity : 0.9991


,Doc 1 (cricket),Doc 2 (football),Doc 3 (baking)
Doc 1 (cricket),1.0000,0.9991,0.9597
Doc 2 (football),0.9991,1.0000,0.9585
Doc 3 (baking),0.9597,0.9585,1.0000


# Part C : To implement word embedding using GloVe

## Step 1 : Loading pretrained GloVe vectors with gensim

**GloVe** (Global Vectors) also gives every word a vector, but it learns differently. Word2Vec moves
along the text one small window at a time. GloVe first counts, over the *whole* text, how often each
pair of words appears together. It then learns vectors so that words that often appear together get
similar vectors.

Counting needs a lot of text to be useful, and our three paragraphs are far too short. So we use
vectors that the Stanford GloVe team already trained on Wikipedia and news text (6 billion words).
`gensim.downloader` downloads them once (about 66 MB) and keeps a copy on disk. They cover 400,000 words,
with 50 numbers per word. Because they were trained on so much text, the closest words now make real sense.

In [4]:
import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-50")

print("vocabulary size       :", len(glove))
print("our words not in GloVe:", [t for doc in tokens for t in doc if t not in glove])
print("vector for 'team'     :", glove["team"][:5].round(3), "... (50 numbers)")
print("closest to 'team'     :", glove.most_similar("team", topn=3))
print("closest to 'dough'    :", glove.most_similar("dough", topn=3))

vocabulary size       : 400000
our words not in GloVe: []
vector for 'team'     : [-0.628  0.123 -0.391  0.879  0.286] ... (50 numbers)
closest to 'team'     : [('teams', 0.9057689905166626), ('football', 0.8386602997779846), ('player', 0.8329677581787109)]
closest to 'dough'    : [('knead', 0.8126067519187927), ('baking', 0.7970367074012756), ('kneading', 0.7907828688621521)]


# Part D : To find similarity between two documents using GloVe

## Step 1 : Averaging GloVe vectors and comparing documents with cosine similarity

This is the same method as Part B, using the same `document_vector` and `cosine_similarity` functions.
The only thing that changes is where the word vectors come from.

The scores are far more spread out than with our small Word2Vec model. The two sports paragraphs are
still almost the same (about 0.97), but the baking paragraph now drops to about 0.5. GloVe already knows
from its training that *team* and *flour* have little to do with each other.

In [5]:
glove_vectors = [document_vector(t, glove) for t in tokens]

glove_similarity = pd.DataFrame([[round(cosine_similarity(a, b), 4) for b in glove_vectors] for a in glove_vectors],
                                index=doc_labels, columns=doc_labels)

print("Doc 1 vs Doc 2 :", round(cosine_similarity(glove_vectors[0], glove_vectors[1]), 4),
      "| gensim n_similarity :", round(float(glove.n_similarity(tokens[0], tokens[1])), 4))
glove_similarity

Doc 1 vs Doc 2 : 0.9711 | gensim n_similarity : 0.9711


,Doc 1 (cricket),Doc 2 (football),Doc 3 (baking)
Doc 1 (cricket),1.0000,0.9711,0.4938
Doc 2 (football),0.9711,1.0000,0.4736
Doc 3 (baking),0.4938,0.4736,1.0000


# Part E : To implement an application of word embeddings : Duplicate question detection

## Step 1 : Scoring question pairs with GloVe

Sites like Quora, and help desks, get the same question asked in many different ways. If we can tell
that a new question means the same as an old one, we can show the old answer instead of writing it again.

Two duplicate questions often share **no words at all**. For example, *"reset my password"* and
*"forgot my login"* have nothing in common, so word counting (Bag of Words or TF-IDF from Lab 2) would
score them 0. Word embeddings solve this, because *password* and *login* have similar vectors.

The method is exactly the one from Part D: clean each question, average its GloVe vectors, and compare
the two averages with cosine similarity. `text_similarity` in `preprocessing.py` does all three steps.
We call a pair a duplicate when it scores 0.7 or more. We chose 0.7 by looking at these examples: the
real duplicates score about 0.75 to 0.85, and the unrelated pair scores about 0.1.

In [6]:
from preprocessing import text_similarity

pairs = [
    ("How do I reset my password?", "I forgot my login, what now?"),
    ("How can I learn Python quickly?", "What is the fastest way to study Python?"),
    ("How do I reset my password?", "What is the best pizza in Mumbai?"),
    ("Which laptop is good for machine learning?", "Best computer for training neural networks?"),
]

pair_scores = pd.DataFrame(pairs, columns=["Question 1", "Question 2"])
pair_scores["Similarity"] = [round(text_similarity(q1, q2, glove), 4) for q1, q2 in pairs]
pair_scores["Duplicate?"] = pair_scores["Similarity"] >= 0.7
pair_scores

,Question 1,Question 2,Similarity,Duplicate?
0,How do I reset my password?,"I forgot my login, what now?",0.7450,True
1,How can I learn Python quickly?,What is the fastest way to study Python?,0.8313,True
2,How do I reset my password?,What is the best pizza in Mumbai?,0.0949,False
3,Which laptop is good for machine learning?,Best computer for training neural networks?,0.8484,True


## Step 2 : Finding the closest question in an FAQ list

A new question is compared with every question in a small FAQ, and the three closest ones are shown.

This is where the method shows its limits, and the right FAQ does not always come first. Averaging mixes
all the words of a short question into one vector. General words such as *help*, *account* and *customer*
are close to many other words, so they pull the average off course. Averaging also ignores word order,
so *"dog bites man"* and *"man bites dog"* get the same vector. A sentence encoder reads the whole sentence
at once (for example `sentence-transformers`), and it fixes both problems. The cosine similarity step stays
the same.

In [7]:
faq = [
    "How do I reset my password?",
    "How can I change my email address?",
    "How do I delete my account?",
    "What payment methods do you accept?",
    "How do I contact customer support?",
    "Where can I download my invoice?",
]

for question in ["I forgot my login details, help", "Can I pay with a credit card?"]:
    ranked = sorted(faq, key=lambda f: -text_similarity(question, f, glove))[:3]
    print("New question :", question)
    for f in ranked:
        print("   ", round(text_similarity(question, f, glove), 4), " ", f)
    print()

New question : I forgot my login details, help
    0.7591   How do I delete my account?
    0.7185   How can I change my email address?
    0.7164   How do I contact customer support?

New question : Can I pay with a credit card?
    0.7568   How do I contact customer support?
    0.7563   What payment methods do you accept?
    0.6429   How do I delete my account?



# Questions of curiosity

## 1. Describe the differences between Word2Vec's CBOW and Skip-Gram architectures.

Both learn word vectors by playing a guessing game with the words inside a small window of text.
The difference is **which way round the guess goes**.

Take the sentence *"the cat sat on the mat"* with the centre word **sat** and a window of 2:

- **CBOW (Continuous Bag of Words)** looks at the surrounding words and guesses the middle one:
  `[the, cat, on, the] → sat`. The context vectors are averaged into one vector before the guess.
- **Skip-Gram** looks at the middle word and guesses each surrounding word one at a time:
  `sat → the`, `sat → cat`, `sat → on`, `sat → the`.

| | CBOW | Skip-Gram |
|---|---|---|
| Input → output | many context words → one centre word | one centre word → each context word |
| Training examples per position | 1 | one for every neighbour (up to 2 × window) |
| Speed | faster | slower |
| Rare words | weaker, because averaging the context blurs them | stronger, because each rare word gets several examples |
| Best suited to | large texts with lots of common words | small texts and rare words |
| gensim setting | `sg=0` (the default) | `sg=1` |


## 2. How are word embeddings trained using Word2Vec?

Word2Vec is a small neural network with one hidden layer, trained on a fake task: *predict which
words appear near each other*. We never use its predictions. We only keep the weights it learns
along the way, and those weights are the word embeddings.

1. **Start with random vectors.** Every word in the vocabulary gets a random vector of size
   `vector_size`, stored in an *input* table. A second *output* table is used to score which words
   could be neighbours.
2. **Slide a window over the text.** For every word, pair it with the words up to `window` positions
   on each side. For Skip-Gram this gives *(centre, context)* pairs such as *(sat, cat)*.
3. **Make a guess.** Multiply the centre word's vector with the output vectors to get a score for
   every word, and turn the scores into probabilities with softmax.
4. **Measure the mistake and correct it.** The true neighbour should have the highest probability.
   Gradient descent shifts both tables a little so that next time the true neighbour gets a higher
   probability and the other words get a lower one.
5. **Repeat** over the whole text for several `epochs`.

Computing softmax over the whole vocabulary is slow, so real Word2Vec uses **negative sampling**.
Instead of scoring every word, it checks the true neighbour and a handful (5 to 20) of random
*"negative"* words. It learns to answer *yes* for the real neighbour and *no* for the random ones.

Why this works: two words that keep appearing next to the same neighbours (*cricket* and *football*
both sit next to *team*, *match*, *score*) receive the same kind of correction again and again, so their
vectors end up pointing in similar directions. When training ends, the output table is thrown away and
the input table is the set of word embeddings.

## 3. What is Cosine Similarity and cosine distance?

**Cosine similarity** measures the angle between two vectors. It ignores how long the vectors are
and only asks whether they point the same way:

$$\text{cosine similarity}(A, B) = \frac{A \cdot B}{\lVert A \rVert \, \lVert B \rVert}$$

The top is the dot product: multiply matching numbers and add them up. The bottom is the two lengths
multiplied together.

| Value | Meaning | Example |
|---|---|---|
| 1 | same direction, very similar | $[1, 2]$ and $[2, 4]$ |
| 0 | at right angles, unrelated | $[1, 0]$ and $[0, 1]$ |
| −1 | opposite directions | $[1, 2]$ and $[-1, -2]$ |

Length is ignored because a long document has a longer vector than a short one about the same topic.
Cosine similarity still scores the two of them as close.

**Cosine distance** is simply the other side of the same number:

$$\text{cosine distance}(A, B) = 1 - \text{cosine similarity}(A, B)$$

It runs from 0 (same direction) to 2 (opposite directions). Here **smaller means more similar**,
which is what methods such as clustering and nearest neighbour search expect from a distance.